In [1]:
from pathlib import Path

DATA_RAW = Path("datasets/raw")
DATA_PROCESSED = Path("datasets/processed")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
SPLITS = ["train", "val", "test"]


### remove blurry and black images

In [2]:
#delete black and blurry images
import shutil
import cv2
import numpy as np

BLACK_PIXEL_THRESHOLD = 0.93  # 93% pixels near black
BLUR_THRESHOLD = 40


In [3]:
def is_mostly_black(image_path: Path, threshold=BLACK_PIXEL_THRESHOLD) -> bool:
    img = cv2.imread(str(image_path))
    if img is None:
        return True  # treat unreadable images as invalid
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # count near-black pixels
    black_pixels = np.sum(gray < 10)  # intensity < 10 = almost black
    total_pixels = gray.size

    ratio = black_pixels / total_pixels
    return ratio >= threshold


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

In [4]:
def is_blurry(image_path, threshold=BLUR_THRESHOLD):
    img = cv2.imread(str(image_path))
    
    if img is None:
        return True

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Laplacian variance (sharpness measure)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()

    return variance < threshold

In [5]:
def is_bad_image(image_path):
    return is_mostly_black(image_path) or is_blurry(image_path)

In [6]:
from collections import defaultdict

stats = defaultdict(lambda: {"black": 0, "blurry": 0, "total": 0, "kept": 0})

for split in SPLITS:
    img_src_dir = DATA_RAW / IMAGES_DIR / split
    lbl_src_dir = DATA_RAW / LABELS_DIR / split

    img_dst_dir = DATA_PROCESSED / IMAGES_DIR / split
    lbl_dst_dir = DATA_PROCESSED / LABELS_DIR / split

    ensure_dir(img_dst_dir)
    ensure_dir(lbl_dst_dir)

    for img_path in img_src_dir.glob("*.jpg"):
        stats[split]["total"] += 1

        name = img_path.stem
        lbl_path = lbl_src_dir / f"{name}.txt"

        if is_mostly_black(img_path):
            print(f"[{split}] Skipping (black): {img_path.name}")
            stats[split]["black"] += 1
            continue

        if is_blurry(img_path):
            print(f"[{split}] Skipping (blurry): {img_path.name}")
            stats[split]["blurry"] += 1
            continue

        shutil.copy2(img_path, img_dst_dir / img_path.name)

        if lbl_path.exists():
            shutil.copy2(lbl_path, lbl_dst_dir / lbl_path.name)
        else:
            print(f"[{split}] Warning: missing label for {img_path.name}")

        stats[split]["kept"] += 1

# ---- summary ----
print("\n===== SUMMARY PER SPLIT =====")
for split, s in stats.items():
    print(
        f"{split}: "
        f"total={s['total']}, "
        f"kept={s['kept']}, "
        f"black={s['black']}, "
        f"blurry={s['blurry']}"
    )

print("\n===== GLOBAL TOTALS =====")
print(
    f"black={sum(s['black'] for s in stats.values())}, "
    f"blurry={sum(s['blurry'] for s in stats.values())}"
)

[train] Skipping (blurry): 0_8102.jpg
[train] Skipping (blurry): 0_8103.jpg
[train] Skipping (blurry): 0_8104.jpg
[train] Skipping (blurry): 0_8105.jpg
[train] Skipping (blurry): 0_8106.jpg
[train] Skipping (blurry): 0_8107.jpg
[train] Skipping (blurry): 0_8108.jpg
[train] Skipping (blurry): 0_8109.jpg
[train] Skipping (blurry): 0_8110.jpg
[train] Skipping (blurry): 0_8111.jpg
[train] Skipping (blurry): 0_8112.jpg
[train] Skipping (blurry): 0_8114.jpg


KeyboardInterrupt: 

In [20]:
print("PreProcessing completed - stats:")

def count_images(base_path):
    counts = {}
    for split in SPLITS:
        path = base_path / IMAGES_DIR / split
        counts[split] = len(list(path.glob("*.jpg")))
    return counts


raw_counts = count_images(DATA_RAW)
processed_counts = count_images(DATA_PROCESSED)



for split in SPLITS:
    before = raw_counts.get(split, 0)
    after = processed_counts.get(split, 0)
    print(f"{split.upper():5s} | before: {before:5d} | after: {after:5d}")

PreProcessing completed - stats:
VAL   | before:  2384 | after:   531
